In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import numpy as np
import warnings
from pathlib import Path
from multiprocessing import Pool, cpu_count

warnings.filterwarnings("ignore")


# ============================================================
# Finite-temperature occupation
# ============================================================

def fermi_occupation(eps, T):
    """
    Stable Fermi occupation:
        f(eps) = 1 / (exp(eps / T) + 1)

    At T = 0, this reduces to theta(-eps).
    """
    eps = np.asarray(eps, dtype=float)

    if T <= 0:
        return np.heaviside(-eps, 0.5)

    x = eps / T
    occ = np.empty_like(x)

    occ[x > 40] = 0.0
    occ[x < -40] = 1.0

    mask = (x >= -40) & (x <= 40)
    occ[mask] = 1.0 / (np.exp(x[mask]) + 1.0)

    return occ


# ============================================================
# Atomic saving
# ============================================================

def atomic_save_npy(file_path, array):
    """
    Atomically update a .npy file.
    This avoids corrupting the file if the job is interrupted during saving.
    """
    file_path = Path(file_path)
    tmp_path = file_path.with_suffix(file_path.suffix + ".tmp")

    with open(tmp_path, "wb") as f:
        np.save(f, array)

    os.replace(tmp_path, file_path)


# ============================================================
# Model + mean-field machinery
# ============================================================

class System:
    def __init__(
        self,
        T,
        L,
        t,
        mu,
        W,
        delta,
        U,
        V,
        disorder,
        disorder1,
        disorder_gamma,
        disorder_eta,
        npf,
        pbc,
        trs,
        W_ratio=0.0,
    ):
        arr = np.ones(L)
        arr[0] = arr[-1] = pbc

        a1 = 1j * np.block([npf[L - 1] * pbc, npf[0:L - 1]])
        a2 = 1j * np.block([npf[1:L], npf[0] * pbc])
        d = 1j * npf[3 * L:4 * L] * arr

        b = 1j * npf[1 * L:2 * L]
        b1 = 1j * np.block([npf[2 * L - 1] * pbc, npf[1 * L:2 * L - 1]])
        b2 = 1j * np.block([npf[1 * L + 1:2 * L], npf[1 * L] * pbc])

        c = 1j * npf[2 * L:3 * L]

        i_gamma_j_gamma_jplus1 = 1j * npf[4 * L:5 * L]
        i_eta_j_eta_jplus1 = 1j * npf[5 * L:6 * L]

        e1 = np.block([i_eta_j_eta_jplus1[-1], i_eta_j_eta_jplus1[0:L - 1]])
        e2 = np.block(
            [
                i_eta_j_eta_jplus1[-2],
                i_eta_j_eta_jplus1[-1],
                i_eta_j_eta_jplus1[0:L - 2],
            ]
        )

        f1 = np.block([i_gamma_j_gamma_jplus1[-1], i_gamma_j_gamma_jplus1[0:L - 1]])
        f2 = i_gamma_j_gamma_jplus1

        H_mu = (
            -1j * (-W * disorder + mu) * np.eye(L, k=0)
            - U * (a1 + a2) * np.eye(L, k=0)
            - V * d * np.eye(L, k=0)
        )

        H_t = 1j * (-t + delta) * np.eye(L, k=1) - U * b * np.eye(L, k=1)

        H_t[L - 1, 0] = (1j * (-t + delta) - U * b[0]) * pbc

        H_t1 = (
            1j * (-t - delta - W * disorder1) * np.eye(L, k=-1)
            + U * c * np.eye(L, k=-1)
            + V * (b1 + b2) * np.eye(L, k=-1)
        )

        H_t1[0, L - 1] = (
            1j * (-t - delta - W * disorder1[0])
            + U * c[L - 1]
            + V * (b1[L - 1] + b2[L - 1])
        ) * pbc

        H_t2 = V * a2 * np.eye(L, k=-2)
        H_t2[0, L - 2] = V * a2[L - 2] * pbc
        H_t2[1, L - 1] = V * a2[L - 1] * pbc

        H_maj_mf = H_mu + H_t + H_t1 + H_t2

        H_gamma0 = (
            trs * np.eye(L, k=1) * (-U * e1 - V * e2)
            + 1j * np.eye(L, k=1) * W * W_ratio * disorder_gamma
        )

        H_eta0 = (
            trs * np.eye(L, k=1) * (-U * f1 - V * f2)
            + 1j * np.eye(L, k=1) * W * W_ratio * disorder_eta
        )

        H_gamma0[L - 1, 0] = (
            trs * (-U * e1[0] - V * e2[0])
            + 1j * W * W_ratio * disorder_gamma[0]
        ) * pbc

        H_eta0[L - 1, 0] = (
            trs * (-U * f1[0] - V * f2[0])
            + 1j * W * W_ratio * disorder_eta[0]
        ) * pbc

        H_gamma = H_gamma0 - H_gamma0.T
        H_eta = H_eta0 - H_eta0.T

        H = np.block(
            [
                [H_gamma, H_maj_mf],
                [H_maj_mf.conj().T, H_eta],
            ]
        )

        self.H = H
        self.L = L

        eigvalsU, eigvecsU = np.linalg.eigh(self.H)

        occ = fermi_occupation(eigvalsU, T)

        self.gamma_eta = np.conj(eigvecsU) @ np.diag(occ) @ eigvecsU.T

        majorana_to_c = np.sqrt(0.5) * np.block(
            [
                [np.eye(L), np.eye(L)],
                [1j * np.eye(L), -1j * np.eye(L)],
            ]
        )

        self.nij = np.conj(majorana_to_c.T) @ self.gamma_eta @ majorana_to_c

        self.E = float(np.real(np.sum(eigvalsU * occ)))

    def calculation_npf(self):
        L = self.L

        gamma_eta = self.gamma_eta[0:L, L:2 * L]

        ai = np.zeros(L)
        bi = np.zeros(L)
        ci = np.zeros(L)
        di = np.zeros(L)
        xi = np.zeros(L)
        yi = np.zeros(L)

        for i in range(L):
            ai[i] = np.real((-1j) * gamma_eta[i, i])
            bi[i] = np.real((1j) * gamma_eta[(i + 1) % L, i])
            ci[i] = np.real((-1j) * gamma_eta[i, (i + 1) % L])
            di[i] = np.real((1j) * gamma_eta[(i + 1) % L, (i - 1) % L])

        gamma_gamma = self.gamma_eta[0:L, 0:L]
        eta_eta = self.gamma_eta[L:2 * L, L:2 * L]

        for i in range(L):
            xi[i] = np.real((-1j) * gamma_gamma[i, (i + 1) % L])
            yi[i] = np.real((-1j) * eta_eta[i, (i + 1) % L])

        return np.block([ai, bi, ci, di, xi, yi])

    def density_matrix(self):
        return self.nij

    def energy(self):
        return self.E


# ============================================================
# CDW and density-spectrum helpers
# ============================================================

def cdw_q_from_ni(n_i):
    """
    CDW component from site densities n_i:
        q = 2 * sum_{k>0} |n(k)|^4 / (sum_{k>0} |n(k)|^2)^2

    The mean is removed.
    The FFT normalization cancels in the ratio.
    """
    n = np.asarray(n_i, dtype=float)
    n = n - n.mean()

    n_k = np.fft.rfft(n)

    p2 = np.abs(n_k[1:]) ** 2
    s2 = p2.sum()

    if s2 == 0:
        return 0.0

    s4 = (p2 ** 2).sum()

    return float(2.0 * s4 / (s2 ** 2))


def density_spectrum(n_i):
    """
    Return:
        k_grid : momentum grid in rad/site on [0, pi]
        Nk_env : |n_k| for mean-removed density
    """
    n = np.asarray(n_i, dtype=float)
    n = n - n.mean()

    n_k = np.fft.rfft(n)

    k_grid = 2.0 * np.pi * np.fft.rfftfreq(len(n), d=1.0)
    Nk_env = np.abs(n_k)

    return k_grid, Nk_env


def _second_derivative_boundary_safe(y, i, dk):
    """
    Boundary-safe finite-difference estimate of f'' at index i.
    """
    N = len(y)

    if N < 3:
        return np.nan

    if (i - 1) >= 0 and (i + 1) < N:
        return (y[i - 1] - 2.0 * y[i] + y[i + 1]) / (dk ** 2)

    if (i + 3) < N:
        return (
            2.0 * y[i]
            - 5.0 * y[i + 1]
            + 4.0 * y[i + 2]
            - y[i + 3]
        ) / (dk ** 2)

    if (i - 3) >= 0:
        return (
            2.0 * y[i]
            - 5.0 * y[i - 1]
            + 4.0 * y[i - 2]
            - y[i - 3]
        ) / (dk ** 2)

    return np.nan


def curvature_at_peak_extremum(k_grid, Nk_env, deg=3, m_points=11, prefer="symmetric"):
    """
    Estimate curvature near the dominant density-spectrum peak.

    Returns:
        curv   : curvature value
        q_idx  : index of the data peak
        Q_data : momentum of the data peak
        Q_fit  : fitted extremum momentum, if available
    """
    k_grid = np.asarray(k_grid, dtype=float)
    y = np.asarray(Nk_env, dtype=float)

    if len(k_grid) != len(y) or len(k_grid) < 3:
        return np.nan, -1, np.nan, np.nan

    if np.all(y[1:] == 0.0):
        return np.nan, -1, np.nan, np.nan

    q_idx = 1 + int(np.argmax(y[1:]))
    Q_data = float(k_grid[q_idx])

    dk = float(k_grid[1] - k_grid[0]) if len(k_grid) > 1 else 1.0

    m_points = max(m_points, deg + 1)

    if prefer == "symmetric":
        half = m_points // 2

        start = max(1, q_idx - half)
        stop = min(len(k_grid), q_idx + half + 1)

        if (stop - start) < (deg + 1) or not (start <= q_idx < stop):
            start = max(1, q_idx - (m_points - 1))
            stop = q_idx + 1
    else:
        start = max(1, q_idx - (m_points - 1))
        stop = q_idx + 1

    idx = np.arange(start, stop)

    if len(idx) < 3:
        curv_fd = _second_derivative_boundary_safe(y, q_idx, dk)

        if np.isfinite(curv_fd):
            return float(curv_fd), q_idx, Q_data, Q_data

        return np.nan, q_idx, Q_data, Q_data

    cur_deg = min(deg, len(idx) - 1)

    if cur_deg < 2:
        curv_fd = _second_derivative_boundary_safe(y, q_idx, dk)

        if np.isfinite(curv_fd):
            return float(curv_fd), q_idx, Q_data, Q_data

        return np.nan, q_idx, Q_data, Q_data

    x = (k_grid[idx] - Q_data) / dk
    yy = y[idx]

    try:
        coeffs = np.polyfit(x, yy, cur_deg)

        p = np.poly1d(coeffs)
        dp = p.deriv(1)

        roots = dp.r

        x_min = float(x[0])
        x_max = float(x[-1])

        real_roots = [
            r.real
            for r in roots
            if abs(getattr(r, "imag", 0.0)) < 1e-12
        ]

        in_window = [
            xr
            for xr in real_roots
            if (x_min - 1e-12) <= xr <= (x_max + 1e-12)
        ]

        if len(in_window) > 0:
            x_star = min(in_window, key=lambda xr: abs(xr))

            curv = float(p.deriv(2)(x_star)) / (dk ** 2)
            Q_fit = float(Q_data + x_star * dk)

            return curv, q_idx, Q_data, Q_fit

        curv_at_Q = float(p.deriv(2)(0.0)) / (dk ** 2)

        return curv_at_Q, q_idx, Q_data, Q_data

    except Exception:
        curv_fd = _second_derivative_boundary_safe(y, q_idx, dk)

        if np.isfinite(curv_fd):
            return float(curv_fd), q_idx, Q_data, Q_data

        return np.nan, q_idx, Q_data, Q_data


# ============================================================
# One disorder realization for one (W, T) point
# ============================================================

def run_one_realization(task):
    (
        i_w,
        i_T,
        WX,
        T_val,
        seed,
        g_fixed,
        L,
        t0,
        delta0,
        mu0,
        VY_to_UY_ratio,
        pbc,
        trs,
        delta_mf,
        fric,
        mf_precision,
        jj_iter,
        W_ratio,
    ) = task

    rng = np.random.default_rng(seed)

    UY = 2.0 * g_fixed
    VY = VY_to_UY_ratio * UY

    disorder = rng.uniform(-1.0, 1.0, size=L)
    disorder1 = rng.uniform(-1.0, 1.0, size=L)
    disorder_gamma = rng.uniform(-1.0, 1.0, size=L)
    disorder_eta = rng.uniform(-1.0, 1.0, size=L)

    npf = delta_mf * rng.uniform(-1.0, 1.0, size=6 * L)

    # Warm start at U = V = 0.
    try:
        sys0 = System(
            T=T_val,
            L=L,
            t=t0,
            mu=mu0,
            W=WX,
            delta=delta0,
            U=0.0,
            V=0.0,
            disorder=disorder,
            disorder1=disorder1,
            disorder_gamma=disorder_gamma,
            disorder_eta=disorder_eta,
            npf=npf,
            pbc=pbc,
            trs=trs,
            W_ratio=W_ratio,
        )

        npf = sys0.calculation_npf()

    except Exception:
        pass

    mf_converged = False
    jj = 0

    while jj < jj_iter:
        sys = System(
            T=T_val,
            L=L,
            t=t0,
            mu=mu0,
            W=WX,
            delta=delta0,
            U=UY,
            V=VY,
            disorder=disorder,
            disorder1=disorder1,
            disorder_gamma=disorder_gamma,
            disorder_eta=disorder_eta,
            npf=npf,
            pbc=pbc,
            trs=trs,
            W_ratio=W_ratio,
        )

        npf_new = sys.calculation_npf()

        eps = float(np.max(np.abs(npf - npf_new)))

        if eps < mf_precision:
            mf_converged = True
            npf = npf_new
            break

        npf = (1.0 - fric) * npf_new + fric * npf

        jj += 1

    if not mf_converged:
        return i_w, i_T, np.nan, False, jj, None

    sys = System(
        T=T_val,
        L=L,
        t=t0,
        mu=mu0,
        W=WX,
        delta=delta0,
        U=UY,
        V=VY,
        disorder=disorder,
        disorder1=disorder1,
        disorder_gamma=disorder_gamma,
        disorder_eta=disorder_eta,
        npf=npf,
        pbc=pbc,
        trs=trs,
        W_ratio=W_ratio,
    )

    nij = sys.density_matrix()

    n_i = np.real(np.diag(nij)[:L])

    cdw_q = cdw_q_from_ni(n_i)

    _, Nk_env = density_spectrum(n_i)

    return i_w, i_T, cdw_q, True, jj, Nk_env


# ============================================================
# Main W-T phase-diagram calculation
# ============================================================

def make_seed(seed0, i_w, i_T, n):
    """
    Deterministic 32-bit seed.
    """
    ss = np.random.SeedSequence([seed0, i_w, i_T, n])
    return int(ss.generate_state(1)[0])


def save_all_outputs(
    out_path,
    L,
    g_fixed,
    WX_range,
    T_values,
    cdw_avg,
    counts,
    active,
    Nk_env_mean,
    curv_at_Q,
    Q_idx_arr,
    Q_val_arr,
    Q_ext_arr,
    Q_avg_arr,
):
    """
    Save all running averaged W-T data products.
    """
    atomic_save_npy(out_path / f"{2 * L}_fixed_g_{g_fixed}_cdw_WT.npy", cdw_avg)
    atomic_save_npy(out_path / f"{2 * L}_fixed_g_{g_fixed}_counts_WT.npy", counts)
    atomic_save_npy(out_path / f"{2 * L}_fixed_g_{g_fixed}_active_WT.npy", active)

    atomic_save_npy(out_path / f"{2 * L}_fixed_g_{g_fixed}_nk_env_mean_WT.npy", Nk_env_mean)
    atomic_save_npy(out_path / f"{2 * L}_fixed_g_{g_fixed}_nk_curvature_WT.npy", curv_at_Q)
    atomic_save_npy(out_path / f"{2 * L}_fixed_g_{g_fixed}_nk_Q_idx_WT.npy", Q_idx_arr)
    atomic_save_npy(out_path / f"{2 * L}_fixed_g_{g_fixed}_nk_Q_value_WT.npy", Q_val_arr)
    atomic_save_npy(out_path / f"{2 * L}_fixed_g_{g_fixed}_nk_Qext_value_WT.npy", Q_ext_arr)
    atomic_save_npy(out_path / f"{2 * L}_fixed_g_{g_fixed}_nk_Qavg_WT.npy", Q_avg_arr)

    atomic_save_npy(out_path / f"{2 * L}_fixed_g_{g_fixed}_W_values.npy", WX_range)
    atomic_save_npy(out_path / f"{2 * L}_fixed_g_{g_fixed}_T_values.npy", T_values)


def run_WT_phase_diagram(
    # Physics / model params
    L=100,
    t0=0.5,
    delta0=0.5,
    int_stagg=1,
    VY_to_UY_ratio=0.0,
    g_fixed=0.45,
    pbc=1,
    trs=0,
    W_ratio=0.0,

    # W-T grid
    WX_range=np.asarray([i**2 / 400 for i in range(21)]),
    T_values=np.linspace(0.0, 2.0, 41),

    # MF loop
    delta_mf=1e-3,
    fric=0.5,
    mf_precision=1e-9,
    jj_iter=800,

    # Disorder averaging
    dis_number_max=1000,
    max_consecutive_fail=20,

    # Runtime
    processes=None,

    # IO
    out_dir="main_fig_CDW_WT_phase_diagram/",
):
    """
    W-T phase diagram at fixed g.

    The scan coordinates are:
        axis 0: W
        axis 1: T

    Main saved arrays:
        cdw_avg[i_W, i_T]
        Nk_env_mean[i_W, i_T, k]
        Q_avg_arr[i_W, i_T]
        curv_at_Q[i_W, i_T]

    All averaged data are updated on disk after each disorder realization.
    """
    mu0 = -int_stagg * (t0 + delta0)

    WN = len(WX_range)
    TN = len(T_values)

    Lk = L // 2 + 1
    k_grid = 2.0 * np.pi * np.fft.rfftfreq(L, d=1.0)

    counts = np.zeros((WN, TN), dtype=int)
    active = np.ones((WN, TN), dtype=bool)
    fail_streak = np.zeros((WN, TN), dtype=int)

    cdw_avg = np.full((WN, TN), np.nan, dtype=float)

    Nk_env_mean = np.full((WN, TN, Lk), np.nan, dtype=float)

    curv_at_Q = np.full((WN, TN), np.nan, dtype=float)
    Q_idx_arr = np.full((WN, TN), -1, dtype=int)
    Q_val_arr = np.full((WN, TN), np.nan, dtype=float)
    Q_ext_arr = np.full((WN, TN), np.nan, dtype=float)

    Q_avg_arr = np.full((WN, TN), np.nan, dtype=float)

    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    if processes is None:
        processes = max(1, cpu_count())

    print(f"Fixed g = {g_fixed}")
    print(f"W-T phase diagram: |W| = {WN}, |T| = {TN}")
    print(f"T range: {T_values[0]} to {T_values[-1]}")
    print(f"Disorder realizations: {dis_number_max}")
    print(f"Processes: {processes}")
    print(f"Output directory: {out_path}")

    seed0 = 123456789

    save_all_outputs(
        out_path=out_path,
        L=L,
        g_fixed=g_fixed,
        WX_range=WX_range,
        T_values=T_values,
        cdw_avg=cdw_avg,
        counts=counts,
        active=active,
        Nk_env_mean=Nk_env_mean,
        curv_at_Q=curv_at_Q,
        Q_idx_arr=Q_idx_arr,
        Q_val_arr=Q_val_arr,
        Q_ext_arr=Q_ext_arr,
        Q_avg_arr=Q_avg_arr,
    )

    for n in range(dis_number_max):
        tasks = []

        for i_w, WX in enumerate(WX_range):
            for i_T, T_val in enumerate(T_values):
                if not active[i_w, i_T]:
                    continue

                seed = make_seed(seed0, i_w, i_T, n)

                tasks.append(
                    (
                        i_w,
                        i_T,
                        WX,
                        T_val,
                        seed,
                        g_fixed,
                        L,
                        t0,
                        delta0,
                        mu0,
                        VY_to_UY_ratio,
                        pbc,
                        trs,
                        delta_mf,
                        fric,
                        mf_precision,
                        jj_iter,
                        W_ratio,
                    )
                )

        if not tasks:
            print("All grid points excluded by MF non-convergence. Stopping.")
            break

        print(f"Realization {n + 1}/{dis_number_max} — active points: {len(tasks)}")

        with Pool(processes=processes) as pool:
            results = pool.map(run_one_realization, tasks)

        for res in results:
            i_w, i_T, cdw_single, mf_ok, jj, Nk_env_single = res

            if not mf_ok or not np.isfinite(cdw_single) or Nk_env_single is None:
                fail_streak[i_w, i_T] += 1

                if fail_streak[i_w, i_T] >= max_consecutive_fail:
                    active[i_w, i_T] = False

                continue

            fail_streak[i_w, i_T] = 0

            kcnt = counts[i_w, i_T]

            if kcnt == 0 or not np.isfinite(cdw_avg[i_w, i_T]):
                cdw_avg[i_w, i_T] = cdw_single
            else:
                cdw_avg[i_w, i_T] += (cdw_single - cdw_avg[i_w, i_T]) / (kcnt + 1)

            if kcnt == 0 or np.any(np.isnan(Nk_env_mean[i_w, i_T])):
                Nk_env_mean[i_w, i_T] = Nk_env_single
            else:
                Nk_env_mean[i_w, i_T] += (
                    Nk_env_single - Nk_env_mean[i_w, i_T]
                ) / (kcnt + 1)

            counts[i_w, i_T] = kcnt + 1

            Y = Nk_env_mean[i_w, i_T]

            # This preserves the diagnostic from your original code:
            #     Q_avg = Y[-1] / L
            # where Y = averaged |n_k|.
            Q_avg_arr[i_w, i_T] = Y[-1] / L

            try:
                curv, q_idx, Q_data, Q_fit = curvature_at_peak_extremum(
                    k_grid,
                    Nk_env_mean[i_w, i_T],
                    deg=3,
                    m_points=11,
                    prefer="symmetric",
                )
            except Exception:
                curv, q_idx, Q_data, Q_fit = np.nan, -1, np.nan, np.nan

            curv_at_Q[i_w, i_T] = curv
            Q_idx_arr[i_w, i_T] = q_idx
            Q_val_arr[i_w, i_T] = Q_data
            Q_ext_arr[i_w, i_T] = Q_fit

        save_all_outputs(
            out_path=out_path,
            L=L,
            g_fixed=g_fixed,
            WX_range=WX_range,
            T_values=T_values,
            cdw_avg=cdw_avg,
            counts=counts,
            active=active,
            Nk_env_mean=Nk_env_mean,
            curv_at_Q=curv_at_Q,
            Q_idx_arr=Q_idx_arr,
            Q_val_arr=Q_val_arr,
            Q_ext_arr=Q_ext_arr,
            Q_avg_arr=Q_avg_arr,
        )

    print("Done.")

    return (
        cdw_avg,
        counts,
        active,
        Nk_env_mean,
        curv_at_Q,
        Q_val_arr,
        Q_ext_arr,
        Q_avg_arr,
    )


if __name__ == "__main__":

    # -------------------------------------------------------------------------
    # Symmetry class
    # -------------------------------------------------------------------------

    # Relative strength of the time-reversal-symmetry-breaking random hopping
    # between Majoranas of the same type.
    #
    # W_ratio = 0:
    #     suppress the random gamma-gamma and eta-eta hopping terms.
    #
    # W_ratio > 0:
    #     introduce explicit TRS-breaking disorder and move the model toward
    #     symmetry class D.
    #
    # Together with trs = 0, the present choice W_ratio = 0 defines the
    # Majorana-Hubbard model in symmetry class BDI.
    W_ratio = 0.0

    # Switch controlling interaction-generated mean-field terms in the
    # gamma-gamma and eta-eta Hamiltonian blocks.
    #
    # trs = 0:
    #     suppress these same-Majorana mean-field terms; use for class BDI.
    #
    # trs = 1:
    #     retain these terms; normally used for the class-D calculation.
    trs = 0

    # -------------------------------------------------------------------------
    # Fixed interaction
    # -------------------------------------------------------------------------

    # Interaction parameter held fixed throughout the W-T scan.
    #
    # Internally, the mean-field interaction coupling is defined as
    #
    #     UY = 2 * g_fixed.
    #
    # With g_fixed = 1.25:
    #
    #     UY = 2.5.
    #
    # Since VY_to_UY_ratio = 0 below, the second interaction coupling is
    #
    #     VY = 0.
    #
    # Thus, the complete phase diagram is calculated at one fixed interaction
    # while disorder W and temperature T are varied.
    g_fixed = 1.25

    # -------------------------------------------------------------------------
    # Disorder grid
    # -------------------------------------------------------------------------

    # Nonuniform grid of disorder amplitudes W.
    #
    # Define
    #
    #     x_i = 2*i/20,    i = 0,...,20,
    #
    # and
    #
    #     W_i = x_i * [1 - exp(-4*x_i**2)].
    #
    # This produces 21 disorder values ranging from W = 0 to approximately
    # W = 2.
    #
    # For weak disorder,
    #
    #     W_i approximately equals 4*x_i**3,
    #
    # so the points are more densely distributed near W = 0. This improves
    # the resolution of the weak-disorder part of the W-T phase diagram.
    WX_range = np.asarray([
        (2*i/20) * (1 - np.exp(-4*(2*i/20)**2))
        for i in range(21)
    ])

    # An alternative quadratic grid over 0 <= W <= 1 is:
    #
    # WX_range = np.asarray([i**2 / 400 for i in range(21)])

    # -------------------------------------------------------------------------
    # Temperature grid
    # -------------------------------------------------------------------------

    # Temperatures included in the W-T phase diagram.
    #
    # This array contains 41 uniformly spaced temperatures over
    #
    #     0 <= T <= 2.
    #
    # The temperature is measured in the same energy units as t, Delta, mu,
    # U, and W, with Boltzmann's constant set to k_B = 1.
    #
    # Therefore, with t0 = 0.5, the highest temperature corresponds to
    #
    #     T / t0 = 4.
    T_values = np.linspace(0.0, 2.0, 41)

    (
        cdw,
        counts,
        active,
        Nk_mean,
        curv,
        Q_vals,
        Qext_vals,
        Qavg_vals,
    ) = run_WT_phase_diagram(

        # ---------------------------------------------------------------------
        # System size and quadratic Hamiltonian
        # ---------------------------------------------------------------------

        # Number of complex-fermion lattice sites.
        #
        # The Hamiltonian is represented in the Majorana basis and therefore
        # has matrix dimension 2L.
        #
        # Here:
        #
        #     L  = 200 complex-fermion sites,
        #     2L = 400 Majorana degrees of freedom.
        #
        # Output filenames therefore begin with 2*L = 400.
        L=200,

        # Uniform nearest-neighbor hopping amplitude t.
        t0=0.5,

        # Uniform p-wave pairing amplitude Delta.
        #
        # The choice
        #
        #     t0 = delta0
        #
        # together with the chemical potential defined below corresponds to
        # the critical quadratic Kitaev chain used as the noninteracting part
        # of the Majorana-Hubbard model.
        delta0=0.5,

        # Dimensionless factor used to define the chemical potential:
        #
        #     mu0 = -int_stagg * (t0 + delta0).
        #
        # With the present values,
        #
        #     mu0 = -1.
        #
        # Despite its name, int_stagg does not select the interaction
        # staggering. The Majorana-Hubbard interaction pattern is selected by
        # VY_to_UY_ratio below.
        int_stagg=1,

        # ---------------------------------------------------------------------
        # Majorana-Hubbard interaction
        # ---------------------------------------------------------------------

        # Ratio between the two interaction couplings:
        #
        #     UY = 2 * g_fixed,
        #     VY = VY_to_UY_ratio * UY.
        #
        # VY_to_UY_ratio = 0:
        #     VY = 0, selecting the Majorana-Hubbard limit with staggered
        #     interaction couplings in the Majorana representation.
        #
        # VY_to_UY_ratio = 1:
        #     VY = UY, corresponding to the fully critical chain with uniform
        #     interaction couplings in the Majorana representation.
        VY_to_UY_ratio=0.0,

        # Interaction value defined above and held fixed for the entire scan.
        #
        # For g_fixed = 1.25:
        #
        #     UY = 2.5,
        #     VY = 0.
        g_fixed=g_fixed,

        # ---------------------------------------------------------------------
        # Boundary conditions and symmetry class
        # ---------------------------------------------------------------------

        # Boundary-condition switch:
        #
        # pbc = 1:
        #     periodic boundary conditions.
        #
        # pbc = 0:
        #     open boundary conditions.
        pbc=1,

        # Suppress interaction-generated gamma-gamma and eta-eta mean fields
        # for the BDI calculation.
        trs=trs,

        # Suppress explicit TRS-breaking gamma-gamma and eta-eta disorder.
        W_ratio=W_ratio,

        # ---------------------------------------------------------------------
        # W-T scan
        # ---------------------------------------------------------------------

        # Disorder values forming axis 0 of all saved W-T arrays.
        #
        # The array index convention is
        #
        #     result[i_W, i_T].
        WX_range=WX_range,

        # Temperature values forming axis 1 of all saved W-T arrays.
        #
        # Each temperature is used directly in the Fermi occupation
        #
        #     f(epsilon,T) = 1 / [exp(epsilon/T) + 1].
        #
        # At T = 0, the code uses
        #
        #     f(epsilon,0) = heaviside(-epsilon, 0.5).
        T_values=T_values,

        # ---------------------------------------------------------------------
        # Initial mean-field configuration
        # ---------------------------------------------------------------------

        # Amplitude of the random initial mean-field configuration.
        #
        # For each disorder realization, the code first generates 6L
        # correlators uniformly in
        #
        #     [-delta_mf, delta_mf].
        #
        # It then attempts to replace this initialization with a warm start
        # obtained from the corresponding noninteracting system at the same
        # values of W and T.
        delta_mf=1e-3,

        # ---------------------------------------------------------------------
        # Self-consistent finite-temperature mean-field iteration
        # ---------------------------------------------------------------------

        # Linear mixing or damping parameter:
        #
        #     npf_new_mixed =
        #         (1 - fric)*npf_calculated + fric*npf_previous.
        #
        # fric = 0:
        #     use the newly calculated mean fields directly.
        #
        # Larger fric:
        #     retain more of the preceding iteration. This can improve
        #     stability but generally slows convergence.
        fric=0.5,

        # Mean-field convergence tolerance.
        #
        # A realization is considered converged when
        #
        #     max(abs(npf_previous - npf_calculated)) < mf_precision.
        mf_precision=1e-9,

        # Maximum number of self-consistency iterations for one disorder
        # realization at one point of the (W,T) grid.
        #
        # If convergence is not reached within jj_iter iterations, that
        # realization is marked as failed and is excluded from the running
        # disorder average.
        jj_iter=800,

        # ---------------------------------------------------------------------
        # Disorder averaging
        # ---------------------------------------------------------------------

        # Maximum number of disorder realizations evaluated at each active
        # point of the (W,T) grid.
        #
        # The disorder-realization loop is sequential. For each realization,
        # all active (W,T) grid points are evaluated in parallel.
        #
        # The random seed depends on
        #
        #     realization index,
        #     W-grid index,
        #     T-grid index.
        #
        # Consequently, different temperatures currently use independent
        # disorder configurations, even at the same value of W.
        dis_number_max=1000,

        # ---------------------------------------------------------------------
        # Treatment of mean-field convergence failures
        # ---------------------------------------------------------------------

        # Maximum number of consecutive mean-field convergence failures
        # allowed at one (W,T) point.
        #
        # Once this number is reached, the point is marked inactive and no
        # further disorder realizations are attempted there.
        #
        # A successful realization resets the failure counter to zero.
        max_consecutive_fail=20,

        # ---------------------------------------------------------------------
        # Parallelization
        # ---------------------------------------------------------------------

        # Number of worker processes used to evaluate active (W,T) points in
        # parallel.
        #
        # processes = None:
        #     use all CPU cores reported by multiprocessing.cpu_count().
        #
        # On a 128-core machine, an explicit value such as
        #
        #     processes=120
        #
        # can be used to reserve several cores for the operating system.
        processes=None,

        # ---------------------------------------------------------------------
        # Output
        # ---------------------------------------------------------------------

        # Directory in which the running disorder-averaged W-T data are saved.
        #
        # All output arrays are updated after every completed disorder
        # realization using atomic file replacement. This reduces the risk of
        # corrupting a file if the calculation is interrupted during saving.
        #
        # The saved files include:
        #
        #     *_cdw_WT.npy
        #         Disorder-averaged CDW diagnostic.
        #
        #     *_counts_WT.npy
        #         Number of successfully accumulated realizations at each
        #         (W,T) point.
        #
        #     *_active_WT.npy
        #         Boolean mask indicating which grid points remain active.
        #
        #     *_nk_env_mean_WT.npy
        #         Disorder-averaged momentum-space density amplitude |n_k|.
        #
        #     *_nk_curvature_WT.npy
        #         Curvature near the dominant density-spectrum peak.
        #
        #     *_nk_Q_idx_WT.npy
        #         Index of the dominant momentum-space peak.
        #
        #     *_nk_Q_value_WT.npy
        #         Momentum of the dominant peak on the discrete data grid.
        #
        #     *_nk_Qext_value_WT.npy
        #         Momentum of the extremum estimated from a local polynomial
        #         fit.
        #
        #     *_nk_Qavg_WT.npy
        #         Diagnostic defined in the current implementation as
        #         Nk_env_mean[..., -1] / L.
        #
        #     *_W_values.npy and *_T_values.npy
        #         The disorder and temperature coordinate arrays.
        out_dir="../mean_field_data/raw/data_CDW_temperature_map_Majorana_Hubbard_chain_BDI/",
    )

Fixed g = 1.25
W-T phase diagram: |W| = 21, |T| = 41
T range: 0.0 to 2.0
Disorder realizations: 1000
Processes: 128
Output directory: main_fig_CDW_WT_phase_diagram
Realization 1/1000 — active points: 861
Realization 2/1000 — active points: 861
Realization 3/1000 — active points: 861
Realization 4/1000 — active points: 861
Realization 5/1000 — active points: 861
Realization 6/1000 — active points: 861
Realization 7/1000 — active points: 861
Realization 8/1000 — active points: 861
Realization 9/1000 — active points: 861


Process ForkPoolWorker-1131:


KeyboardInterrupt: 